# Section 07: 问答系统（Extractive QA）核心总结

## 任务定义
**抽取式问答（Extractive QA）** = 给定问题 + 上下文段落，从段落中找出答案的起始和结束位置。

**输出不是生成的文本，而是两个整数 (start_pos, end_pos)**，指向原文中答案的 token 范围。

## 本节任务
在 **SQuAD** 数据集上微调 `bert-base-cased`。

## 核心挑战
QA 任务的两大难点：
1. **上下文可能超过模型最大长度**（BERT 最大 512 token），需要**滑动窗口**处理
2. **后处理复杂**：从多个窗口的 logit 中找出最佳答案，并转换回原文字符位置

## 完整流程
```
SQuAD（问题 + 上下文 + 答案字符位置）
    ↓ tokenize（问题+上下文，stride 滑动窗口）
    ↓ offset_mapping（token位置→字符位置）
    ↓ 标签：找出答案文本的 start/end token index
    ↓ AutoModelForQuestionAnswering（输出 start_logits, end_logits）
    ↓ 后处理：多窗口 → 最佳答案 → 字符级文本
    ↓ 评估：Exact Match (EM) + F1
```

---
## 第一步：数据集结构

In [9]:
from datasets import load_dataset

raw_datasets = load_dataset("squad")
print(raw_datasets)

# 每条样本包含：问题、上下文段落、答案（字符级起始位置+文本）
print("Context:", raw_datasets["train"][0]["context"][:100])
print("Question:", raw_datasets["train"][0]["question"])
print("Answer:", raw_datasets["train"][0]["answers"])
# answers = {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}
# answer_start 是字符级索引，而非 token 索引！

# 注意：验证集可能有多个参考答案
print("\n验证集多答案示例:", raw_datasets["validation"][2]["answers"])
# {'text': ['Santa Clara, California', "Levi's Stadium", ...], 'answer_start': [403, 355, ...]}

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}

验证集多答案示例: {'text': ['Santa Clara, California', "Levi's Stadium", "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."], 'answer_start': [403, 355, 355]}


---
## 第二步（核心难点）：滑动窗口 + offset_mapping

### 问题：上下文超长
问题通常很短（~20 token），但上下文可能很长（>512 token）。
BERT 最大输入 512 token，超出部分会被截断 → **答案可能被截掉！**

### 解决方案：滑动窗口切分
```
[CLS] 问题 [SEP] 上下文1-350 [SEP]    ← 窗口1
[CLS] 问题 [SEP] 上下文200-512 [SEP]  ← 窗口2（与窗口1重叠150 token = stride）
[CLS] 问题 [SEP] 上下文400-600 [SEP]  ← 窗口3
```

### offset_mapping：token 位置 → 字符位置
答案标注是字符级别的（`answer_start=515`），但模型处理 token 级别。
`offset_mapping` 告诉我们每个 token 对应原文的哪个字符范围。

In [10]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

context  = raw_datasets["train"][0]["context"]
question = raw_datasets["train"][0]["question"]

# 演示滑动窗口切分
inputs = tokenizer(
    question, context,
    max_length=100,
    truncation="only_second",      # 只截断第二个输入（上下文），不截断问题
    stride=50,                     # 相邻窗口重叠 50 token
    return_overflowing_tokens=True,# 超长上下文自动切多个窗口
    return_offsets_mapping=True,   # 返回每个 token 的字符偏移量
)

print(f"该样本被切成了 {len(inputs['input_ids'])} 个窗口")
print(f"\noffset_mapping 示例（前10个token的字符范围）:")
print(inputs["offset_mapping"][0][:10])
# [(0,0), (0,2), (3,5), ...] — (0,0) 表示特殊 token（无对应字符）

该样本被切成了 4 个窗口

offset_mapping 示例（前10个token的字符范围）:
[(0, 0), (0, 2), (3, 7), (8, 11), (12, 15), (16, 22), (23, 27), (28, 37), (38, 44), (45, 47)]


---
## 第三步：训练集预处理 — 找出答案的 token 位置

In [11]:
max_length = 384   # 每个窗口最多 384 个 token（BERT 上限 512，留余量给特殊 token）
stride = 128       # 相邻窗口重叠 128 token，防止答案恰好落在窗口边界被截断

def preprocess_training_examples(examples):
    """
    训练集预处理：将原始 QA 样本转换为模型可训练的格式。

    输入 examples（一个 batch）:
      examples["question"] = ["To whom did the Virgin Mary appear...", ...]
      examples["context"]  = ["Architecturally, the school has ...", ...]
      examples["answers"]  = [{"text": ["Saint Bernadette"], "answer_start": [515]}, ...]

    输出:
      inputs dict，新增两个字段：
        "start_positions": [int, ...]  ← 每个窗口中答案起始 token 的索引
        "end_positions":   [int, ...]  ← 每个窗口中答案结束 token 的索引
      若答案不在该窗口内，两者均为 0（指向 [CLS]，告诉模型"此窗口无答案"）
    """
    questions = [q.strip() for q in examples["question"]]

    # ── Step 1：分词 + 滑动窗口切分 ──────────────────────────────────────────
    # tokenizer 同时处理 (question, context) 对，输出已拼接为：
    #   [CLS] 问题 tokens [SEP] 上下文 tokens [SEP] [PAD]...
    #
    # 关键参数：
    #   truncation="only_second"      → 只截断上下文（第2个输入），问题不被截断
    #   return_overflowing_tokens=True → 上下文超长时自动生成多个窗口
    #   return_offsets_mapping=True    → 返回每个 token 对应原文的字符范围
    #   padding="max_length"           → 不足 384 的用 [PAD] 补齐
    inputs = tokenizer(
        questions, examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    # 此时 inputs["input_ids"] 的 shape：(num_windows, 384)
    # 若 batch 中某条上下文很长，它会变成多个窗口，num_windows >= len(examples)
    #
    # 示例（1条样本被切成2个窗口）：
    #   inputs["input_ids"]  → [[101, 2000, 1169, ..., 0, 0], [101, 2000, 1169, ..., 0, 0]]
    #   len(inputs["input_ids"]) = 2（而原始样本只有 1 条）

    # ── Step 2：提取辅助映射，从 inputs 中移除（不传给模型）─────────────────
    offset_mapping = inputs.pop("offset_mapping")
    # offset_mapping[i] = [(0,0), (0,2), (3,7), ..., (0,0)]
    #   每个元素是 (char_start, char_end)，表示该 token 对应原文字符的范围
    #   特殊 token（[CLS]/[SEP]/[PAD]）的偏移为 (0, 0)

    sample_map = inputs.pop("overflow_to_sample_mapping")
    # sample_map[i] = j  ← 第 i 个窗口来自第 j 个原始样本
    # 示例（3条样本，第2条很长被切成2个窗口）：
    #   sample_map = [0, 1, 1, 2]
    #   窗口0 → 样本0，窗口1&2 → 样本1，窗口3 → 样本2

    answers = examples["answers"]
    # answers[j] = {"text": ["Saint Bernadette Soubirous"], "answer_start": [515]}
    # answer_start 是字符级索引（而不是 token 索引！）

    start_positions = []
    end_positions   = []

    # ── Step 3：为每个窗口计算答案的 start/end token 索引 ────────────────────
    for i, offset in enumerate(offset_mapping):
        # offset = [(0,0),(0,2),(3,7),...] 长度为 384
        # 通过 sample_map 找到该窗口对应的原始样本
        sample_idx = sample_map[i]
        answer     = answers[sample_idx]

        # 答案在原文中的字符范围 [start_char, end_char)
        start_char = answer["answer_start"][0]             # e.g. 515
        end_char   = start_char + len(answer["text"][0])   # e.g. 515 + 26 = 541

        # sequence_ids 标记每个 token 属于哪个输入序列：
        #   None → [CLS], [SEP], [PAD] 等特殊 token
        #   0    → 问题 tokens
        #   1    → 上下文 tokens
        # 示例：[None,0,0,0,...,0,None,1,1,...,1,None,None,...,None]
        #        [CLS] 问题         [SEP] 上下文        [SEP] [PAD]
        sequence_ids = inputs.sequence_ids(i)

        # 找到上下文在该窗口中的 token 范围 [context_start, context_end]
        # 即 sequence_ids 中第一个 1 到最后一个 1 的位置
        idx = 0
        while sequence_ids[idx] != 1: idx += 1
        context_start = idx                     # 上下文第一个 token 的索引
        while sequence_ids[idx] == 1: idx += 1
        context_end = idx - 1                   # 上下文最后一个 token 的索引
        # 示例：问题有 15 个 token，则 context_start=16，context_end 可能是 300

        # ── 检查答案是否在该窗口内 ──────────────────────────────────────────
        # offset[context_start][0]：上下文第一个 token 的字符起始位置
        # offset[context_end][1]  ：上下文最后一个 token 的字符结束位置
        #
        # 若答案字符范围超出当前窗口覆盖的字符范围 → 答案不在此窗口
        if (offset[context_start][0] > start_char or    # 窗口起点在答案之后
            offset[context_end][1]   < end_char):       # 窗口终点在答案之前
            # 标为 (0, 0) → 指向 [CLS] token
            # 意义：告诉模型"这个窗口里没有答案"
            start_positions.append(0)
            end_positions.append(0)
        else:
            # ── 在窗口内：定位答案的 start token ────────────────────────────
            # 目标：找到包含 start_char 的 token
            # 策略：从 context_start 向右扫，直到 token 的左边界 > start_char
            #        此时退一步（idx-1）即是答案的起始 token
            #
            # 示例：start_char=515，offset 片段如下：
            #   idx=20: (510, 513)  offset[20][0]=510 <= 515 → 继续
            #   idx=21: (514, 520)  offset[21][0]=514 <= 515 → 继续
            #   idx=22: (521, 525)  offset[22][0]=521 > 515  → 停止
            # → start_positions.append(22 - 1) = 21（token 21 包含字符515）
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char: idx += 1
            start_positions.append(idx - 1)

            # ── 在窗口内：定位答案的 end token ──────────────────────────────
            # 目标：找到包含 end_char 的 token
            # 策略：从 context_end 向左扫，直到 token 的右边界 < end_char
            #        此时退一步（idx+1）即是答案的结束 token
            #
            # 示例：end_char=541，offset 片段如下：
            #   idx=25: (536, 542)  offset[25][1]=542 >= 541 → 继续向左
            #   idx=24: (530, 535)  offset[24][1]=535 < 541  → 停止
            # → end_positions.append(24 + 1) = 25（token 25 包含字符541）
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char: idx -= 1
            end_positions.append(idx + 1)

    # ── Step 4：将标签写回 inputs ─────────────────────────────────────────────
    inputs["start_positions"] = start_positions
    inputs["end_positions"]   = end_positions
    # 最终 inputs 包含：
    #   "input_ids":        (num_windows, 384)  ← token id 序列
    #   "attention_mask":   (num_windows, 384)  ← 1=真实token，0=padding
    #   "token_type_ids":   (num_windows, 384)  ← 0=问题，1=上下文（BERT专用）
    #   "start_positions":  (num_windows,)      ← 答案起始 token 索引（或0）
    #   "end_positions":    (num_windows,)      ← 答案结束 token 索引（或0）
    return inputs

train_dataset = raw_datasets["train"].map(
    preprocess_training_examples, batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print(f"原始训练样本: {len(raw_datasets['train'])} → 切窗口后: {len(train_dataset)}")
# 87599 → 88729（多了约 1130 条，因为部分长上下文被切成多个窗口）
# 多出的窗口中大多数 start/end positions 均为 0（答案不在该窗口内）

原始训练样本: 87599 → 切窗口后: 88729


---
## 第四步：验证集预处理（与训练集不同）

验证集预处理的目标不同：
- 不需要计算标签（start/end positions）
- 需要保留 `offset_mapping` 和 `example_id`，供**后处理**时从 logit 还原答案文本

In [12]:
def preprocess_validation_examples(examples):
    """
    验证集预处理：与训练集的核心区别在于「目标不同」。

    训练集目标：计算 start/end token 索引作为监督标签
    验证集目标：保留足够信息供「后处理」使用——
               后处理需要把模型输出的 logit 位置还原回原文字符串

    因此验证集预处理需要保留两个额外字段（训练集不需要）：
      1. "offset_mapping"  ← 只保留上下文部分的字符偏移，问题/特殊token位置置 None
                             后处理用它将 token 索引转换回原文字符范围
      2. "example_id"      ← 记录每个窗口来自哪道原始问题
                             后处理需要把同一道题的多个窗口的 logit 汇总后再选最优答案

    输入 examples（一个 batch）:
      examples["question"] = ["Which NFL team...", ...]
      examples["context"]  = ["Super Bowl 50 was...", ...]
      examples["id"]       = ["56be4db0acb8001400a502ec", ...]  ← 原始问题唯一ID

    输出:
      inputs dict，相比训练集：
        ✗ 无 start_positions / end_positions（验证集不需要标签）
        ✓ 保留修改后的 offset_mapping（问题部分替换为 None）
        ✓ 新增 example_id 字段
    """
    questions = [q.strip() for q in examples["question"]]

    # ── Step 1：分词 + 滑动窗口（与训练集完全相同）────────────────────────────
    inputs = tokenizer(
        questions, examples["context"],
        max_length=max_length,          # 384
        truncation="only_second",
        stride=stride,                  # 128
        return_overflowing_tokens=True,
        return_offsets_mapping=True,    # ← 验证集必须保留，后处理依赖它还原答案文本
        padding="max_length",
    )
    # inputs["offset_mapping"] 初始状态（切窗口后，第0个窗口示例）：
    #   [(0,0), (0,5), (6,10), ..., (0,0), (0,12), (13,18), ..., (0,0), (0,0)]
    #    [CLS]  问题token...         [SEP]  上下文token...          [SEP]  [PAD]
    # 问题和上下文的偏移都有，但后处理只关心上下文部分

    # ── Step 2：移除 sample_map，构建 example_id ─────────────────────────────
    sample_map = inputs.pop("overflow_to_sample_mapping")
    # sample_map[i] = j ← 第 i 个窗口来自第 j 个原始样本（与训练集相同）

    example_ids = []

    # ── Step 3：逐窗口处理 offset_mapping，将非上下文部分置为 None ───────────
    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]

        # 记录该窗口对应的原始问题 ID（字符串形式）
        # 用途：后处理时，按 example_id 将同一道题的所有窗口分组
        # 示例：example_ids = ["56be4db0...", "56be4db0...", "3bfb5e0a...", ...]
        #                       ↑ 同一道题的两个窗口            ↑ 下一道题
        example_ids.append(examples["id"][sample_idx])

        # sequence_ids[k] 标记第 k 个 token 的归属：
        #   None → [CLS], [SEP], [PAD]
        #   0    → 问题 token
        #   1    → 上下文 token
        sequence_ids = inputs.sequence_ids(i)

        offset = inputs["offset_mapping"][i]
        # 关键变换：将 offset_mapping 中所有「非上下文」位置替换为 None
        # 变换前（原始 offset，问题和上下文都有字符偏移）：
        #   [(0,0), (0,5), (6,9), (10,14), (0,0), (0,4), (5,10), ..., (0,0), (0,0)]
        #   [CLS]   问题    问题    问题    [SEP]  上下文  上下文         [SEP]  [PAD]
        #
        # 变换后（只有 sequence_id==1 的上下文 token 保留偏移，其余为 None）：
        #   [None,  None,  None,  None,  None,  (0,4), (5,10), ..., None,  None ]
        #   [CLS]   问题    问题    问题   [SEP]  上下文  上下文        [SEP]  [PAD]
        #
        # 后处理中，None 表示「此 token 不可能是答案的起点或终点」，直接跳过
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(offset)
        ]

    # ── Step 4：写回 example_id，返回 ────────────────────────────────────────
    inputs["example_id"] = example_ids
    # 最终 inputs 包含：
    #   "input_ids":       (num_windows, 384)   ← token id 序列
    #   "attention_mask":  (num_windows, 384)   ← 1=真实token，0=padding
    #   "token_type_ids":  (num_windows, 384)   ← 0=问题，1=上下文
    #   "offset_mapping":  (num_windows, 384)   ← 上下文token为(char_s,char_e)，其余为 None
    #   "example_id":      (num_windows,)       ← 每个窗口对应的原始问题ID字符串
    return inputs

In [12]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples, batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
print(f"原始验证样本: {len(raw_datasets['validation'])} → 切窗口后: {len(validation_dataset)}")
# 10570 → 10822（同样因为部分长上下文被切成多个窗口而增多）
# 注意：验证集保留了 offset_mapping 和 example_id，而训练集的这两个字段已被 pop

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

原始验证样本: 10570 → 切窗口后: 10822


---
## 第五步（最复杂）：后处理 — 从 logit 还原答案

模型输出的是 `start_logits` 和 `end_logits`，每个 token 都有分数。
后处理需要：
1. 对每道题，收集**所有窗口**的 logit
2. 枚举所有合法的 (start, end) 组合（start ≤ end，长度 ≤ max_answer_length）
3. 选分数最高的组合
4. 用 `offset_mapping` 将 token 位置转换回字符位置，截取原文

In [13]:
import evaluate
import numpy as np
import collections
from tqdm.auto import tqdm

n_best = 20             # 每个窗口取 top-20 的 start 和 end 候选位置
max_answer_length = 30  # 答案最大长度（token 数），过滤掉不合理的长答案

def compute_metrics(start_logits, end_logits, features, examples):
    """
    完整的 QA 后处理流程：将模型输出的 logit 还原为可评估的答案文本。

    为什么后处理复杂？
      模型对每个窗口独立输出 start_logits 和 end_logits，但：
        1. 一道题可能有多个窗口，需要跨窗口汇总候选答案
        2. logit 是 token 级别的索引，需要通过 offset_mapping 还原为原文字符串
        3. 需要过滤非法的 (start, end) 组合

    参数：
      start_logits: np.ndarray, shape (num_features, seq_len=384)
                    每个窗口中每个 token 是答案起点的原始分数
      end_logits:   np.ndarray, shape (num_features, seq_len=384)
                    每个窗口中每个 token 是答案终点的原始分数
      features:     验证集（切窗口后），含 "offset_mapping", "example_id"
                    len(features) == num_features（窗口总数）
      examples:     原始验证集，含 "id", "context", "answers"
                    len(examples) == 原始问题数（< num_features）
    """

    # ── Step 1：建立「问题 ID → 窗口索引列表」的映射 ─────────────────────────
    # 目的：对每道题，快速找到它对应的所有窗口（features 中的行号）
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)
    # 示例（某道题被切成3个窗口，feature 索引为 5,6,7）：
    #   example_to_features["56be4db0acb8001400a502ec"] = [5, 6, 7]
    #   example_to_features["3bfb5e0a99c2dc6f49c6d5d3"] = [8]
    # 之后遍历每道原始问题时，就能通过这个 dict 取出所有相关窗口

    predicted_answers = []

    # ── Step 2：逐道题处理，收集所有窗口的候选答案 ───────────────────────────
    for example in tqdm(examples):
        example_id = example["id"]
        context    = example["context"]  # 原文字符串，用于最终截取答案文本
        answers    = []                  # 该题所有窗口产生的候选答案列表

        # 遍历该题对应的每个窗口
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]  # shape: (384,)
            end_logit   = end_logits[feature_index]    # shape: (384,)
            offsets     = features[feature_index]["offset_mapping"]
            # offsets[k]: 上下文 token → (char_start, char_end)，问题/特殊token → None

            # ── 取 top-n_best 的候选索引 ─────────────────────────────────────
            # np.argsort 返回从小到大排序的索引，[-1:-n_best-1:-1] 取最后 n_best 个（从大到小）
            # 示例（seq_len=384，n_best=20）：
            #   start_logit = [0.1, -0.3, 2.5, 1.8, ...]
            #   np.argsort(start_logit) = [..., 3, 2]   ← 最后两个是得分最高的
            #   start_indexes = [2, 3, 7, 15, ...]      ← top-20 token 索引（按分数降序）
            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes   = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            # 此时有 20×20=400 个候选 (start, end) 组合，需要逐一过滤

            for start_index in start_indexes:
                for end_index in end_indexes:

                    # 过滤1：token 对应 None → 该位置是问题 token 或特殊 token
                    #        答案只能来自上下文，直接跳过
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue

                    # 过滤2：end < start → 无意义的逆序组合
                    #        end - start + 1 > max_answer_length → 答案太长（超过30 token）
                    if (end_index < start_index or
                            end_index - start_index + 1 > max_answer_length):
                        continue

                    # ── 通过 offset_mapping 将 token 位置转换为原文字符串 ─────
                    # offsets[start_index][0]：答案起始 token 的字符起始位置
                    # offsets[end_index][1]  ：答案结束 token 的字符结束位置
                    # 示例：
                    #   offsets[21] = (514, 520)，offsets[25] = (536, 542)
                    #   context[514:542] = "Saint Bernadette Soubirous"  ← 答案文本
                    #
                    # logit_score = start 分数 + end 分数（联合得分，用于多候选排序）
                    answers.append({
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    })
        # 该题所有窗口处理完后，answers 可能包含数十到数百个候选，例如：
        #   [{"text": "Saint Bernadette Soubirous", "logit_score": 18.3},
        #    {"text": "Bernadette Soubirous",        "logit_score": 15.7},
        #    {"text": "the Virgin Mary",             "logit_score": 12.1}, ...]

        # ── Step 3：选得分最高的候选作为最终预测答案 ─────────────────────────
        if answers:
            best = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append({"id": example_id, "prediction_text": best["text"]})
        else:
            # 极少情况：所有候选都被过滤掉（如上下文全是特殊 token）
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    # ── Step 4：与参考答案对比，计算 EM 和 F1 ────────────────────────────────
    # theoretical_answers 格式符合 squad metric 的要求：
    #   [{"id": "56be4db0...", "answers": {"text": ["Saint Bernadette Soubirous"], "answer_start": [515]}}, ...]
    theoretical_answers = [
        {"id": ex["id"], "answers": ex["answers"]} for ex in examples
    ]
    # predicted_answers 格式：
    #   [{"id": "56be4db0...", "prediction_text": "Saint Bernadette Soubirous"}, ...]

    metric = evaluate.load("squad")
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)
    # 返回：{'exact_match': 81.18, 'f1': 88.67}
    # exact_match：预测文本与任一参考答案完全一致的比例（大小写/标点归一化后）
    # f1：预测文本与参考答案在词级别的重叠率（允许部分匹配，对同义短答案更友好）

---
## 第六步：训练

In [14]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

# QA 模型：BERT-base-cased + 两个线性层（输出 start_logits 和 end_logits）
# 参数量约 110M，24GB 显存对 BERT-base 非常宽松
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# ── 4090 单卡（24GB）参数说明 ────────────────────────────────────────────────
# BERT-base + seq_len=384 的显存占用估算：
#   batch_size=16：~10GB（激活值 + 梯度 + Adam 优化器状态）
#   batch_size=32：~18GB（仍在安全范围内）
#
# 有效批大小（effective batch size）= per_device_train_batch_size × gradient_accumulation_steps
#   此处 16 × 2 = 32，与 SQuAD 论文推荐一致
#
# bf16 vs fp16：
#   4090 是 Ampere 架构，原生支持 BF16（数值范围更大，训练更稳定，不易溢出）
#   优先用 bf16=True，而非 fp16=True
args = TrainingArguments(
    output_dir="bert-finetuned-squad",

    # ── 批大小与梯度累积 ──────────────────────────────────────────────────────
    per_device_train_batch_size=16,     # 显存占用约 10GB，留有余量；可尝试调至 32
    per_device_eval_batch_size=32,      # 评估不需要梯度，显存占用约一半，可开大
    gradient_accumulation_steps=2,      # 有效批大小 = 16 × 2 = 32

    # ── 精度 ──────────────────────────────────────────────────────────────────
    bf16=True,                          # 4090（Ampere）推荐用 bf16，比 fp16 更稳定
    # fp16=True,                        # 若遇到 bf16 不支持的旧版环境再改回 fp16

    # ── 学习率与调度 ──────────────────────────────────────────────────────────
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,                  # 前 6% 步线性 warmup，防止训练初期梯度爆炸

    # ── 训练周期 ──────────────────────────────────────────────────────────────
    num_train_epochs=3,

    # ── 评估与保存 ────────────────────────────────────────────────────────────
    # eval_strategy="epoch" 不可用：验证集无 start/end positions 标签，
    # Trainer 无法计算 eval_loss，metrics 为空，load_best_model_at_end 会报 KeyError
    # QA 评估需要复杂后处理，在 trainer.train() 完成后手动调用 compute_metrics
    eval_strategy="no",
    save_strategy="epoch",

    # ── 数据加载 ──────────────────────────────────────────────────────────────
    dataloader_num_workers=4,           # 多进程加载数据，减少 GPU 等待

    # ── Hub ───────────────────────────────────────────────────────────────────
    push_to_hub=False,                  # 本地训练关闭；需要上传时改为 True
)

# 注意：QA 任务用普通 Trainer（不需要 Seq2SeqTrainer）
# Trainer 自动计算训练 loss = CE(start_logits, labels) + CE(end_logits, labels)
# compute_metrics 不传给 Trainer，而是在 predict 后手动调用（需要复杂的后处理）
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

trainer.train()
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, raw_datasets["validation"])
# 预期结果：{'exact_match': ~81, 'f1': ~88.5}

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized becaus

Step,Training Loss
500,3.369010
1000,1.448590
1500,1.231336
2000,1.166229
2500,1.104143
3000,0.985035
3500,0.884642
4000,0.872859
4500,0.852970
5000,0.843357


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/10570 [00:00<?, ?it/s]

{'exact_match': 80.66225165562913, 'f1': 88.17357704467621}

In [15]:
from huggingface_hub import notebook_login

# 登录 HuggingFace Hub（会弹出 token 输入框，需提前在 hf.co/settings/tokens 创建 write token）
notebook_login()

# 将模型、tokenizer 和训练配置一起推送到 Hub
# repo_id 默认为 {your_username}/{output_dir}，即 {username}/bert-finetuned-squad
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/goosmanlei/bert-finetuned-squad/commit/8ae19eb8d10ad2bcbfce0866adb4b434efbebb2a', commit_message='End of training', commit_description='', oid='8ae19eb8d10ad2bcbfce0866adb4b434efbebb2a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/goosmanlei/bert-finetuned-squad', endpoint='https://huggingface.co', repo_type='model', repo_id='goosmanlei/bert-finetuned-squad'), pr_revision=None, pr_num=None)

---
## 第七步：推理

In [16]:
from transformers import pipeline

model_checkpoint = "goosmanlei/bert-finetuned-squad"
question_answerer = pipeline("question-answering", model=model_checkpoint)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries —
Jax, PyTorch and TensorFlow — with a seamless integration between them.
"""
question = "Which deep learning libraries back 🤗 Transformers?"

result = question_answerer(question=question, context=context)
print(result)
# {'score': 0.998, 'start': 78, 'end': 105, 'answer': 'Jax, PyTorch and TensorFlow'}

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

---
## 总结

### 核心知识点速查

| 概念 | 说明 |
|------|------|
| 抽取式 QA | 输出答案在上下文中的 start/end token index，而非生成文本 |
| 滑动窗口 | `stride + return_overflowing_tokens=True`，应对超长上下文 |
| `truncation="only_second"` | 只截断上下文（第2输入），问题（第1输入）不截断 |
| `offset_mapping` | token → 字符偏移，用于从 token 位置还原原文片段 |
| `overflow_to_sample_mapping` | 窗口 → 原始样本的映射，一道题可对应多个窗口 |
| `sequence_ids()` | 返回每个 token 属于哪个序列（0=问题，1=上下文，None=特殊token）|
| 后处理 | 枚举 n_best² 个候选，过滤非法答案，取 logit 分数最高者 |
| 评估指标 | Exact Match (EM)：完全匹配率；F1：词级别重叠（允许部分匹配）|

### 模型输出结构
```
AutoModelForQuestionAnswering 输出：
  outputs.start_logits: shape (batch_size, seq_len)  ← 每个 token 是答案起点的分数
  outputs.end_logits:   shape (batch_size, seq_len)  ← 每个 token 是答案终点的分数
  outputs.loss:         标量（训练时）
```

### 训练标签 vs 推理后处理的不对称性
```
训练：
  标签 = (start_position, end_position)，即答案 token 的索引
  loss = CE(start_logits, start_labels) + CE(end_logits, end_labels)
  若答案不在窗口内 → 标签为 (0, 0)，即 [CLS] 位置

推理：
  遍历多个窗口的所有 (start, end) 组合
  score = start_logit[i] + end_logit[j]
  过滤条件：offset 不为 None，start ≤ end，长度 ≤ 30
  用 offset_mapping 从字符位置截取原文
```

### 与其他任务的对比
| 任务 | 特殊处理 | 评估 |
|------|---------|------|
| Token Classification | 标签对齐(word_ids) | seqeval F1 |
| 翻译/摘要 | as_target_tokenizer, generate() | BLEU / ROUGE |
| **问答** | **滑动窗口+offset_mapping+复杂后处理** | **EM + F1** |